In [1]:
import torch
from torch import nn
from torch.nn import functional as F
import polars as pl
import numpy as np
from sklearn.preprocessing import StandardScaler
from typing import Optional, Tuple, Callable
from unicodedata import bidirectional

In [2]:
# ==========================================
# 0 Hyperparameters
# ==========================================

MAX_RUL = 130
WINDOW_SEQ = 28
BATCH_SIZE = 128
EPOCHS = 60

NOISE_STD = 1e-3
DEVICE = torch.device("cuda")


GRAD_CLIP = 1.0

#Model
HIDDEN_SIZE = 80
NUM_LAYERS = 2
CNN_FILTERS = 80
CNN_KERNEL = 3
MLP_DROPOUT = 0.5
LSTM_DROPOUT = 0.3
C1 = 34
C2 = 20
C3 = 10
USE_RESIDUAL = True
INCEPTION_OUT = 82
ATTENTION_HEAD = 4
ATTENTION_DROPOUT = 0.2
D_FF = 256
FF_DROPOUT = 0.3

NUM_OF_WORKERS = 0
LR = 1e-3
WEIGHT_DECAY = 5e-4

T0            = 30        # cosine annealing period
T_MULT        = 2          # cosine annealing multiplier


In [3]:
# ==========================================
# 1 LOAD DATA
# ==========================================
def _validate_join(test_df, test_labels_df):
    n_units_data = test_df.select(["file_path", "unit"]).unique().height
    n_units_labels = test_labels_df.height
    if n_units_data != n_units_labels:
        raise ValueError(
            f"Unit count mismatch after join: {n_units_data} units in test data "
            f"vs {n_units_labels} rows in RUL labels. Check file globbing / ordering."
        )
    if test_df["RUL"].null_count() > 0:
        missing = (
            test_df.filter(pl.col("RUL").is_null())
            .select(["file_path", "unit"])
            .unique()
        )
        raise ValueError(f"RUL join produced nulls for units:\n{missing}")

def _dataset_id_expr(col: str = "file_path") -> pl.Expr:
    """Extract 'FD001' / 'FD002' / etc. from a path regardless of the
    'train_' / 'test_' / 'RUL_' prefix, so different scans can be joined."""
    return pl.col(col).str.extract(r"(FD00\d)", 1).alias("dataset_id")

col_names = ["unit", "cycle"] + [f'op_{i}' for i in range(3)] + [f"s_{i}" for i in range(21)]
drop_cols = ["op_0", "op_1", "op_2", "s_0", "s_4","s_5", "s_9", "s_15", "s_17", "s_18"]
feature_cols = [c for c in col_names if c not in ["unit", "cycle"] + drop_cols]

def load_all_data():
    train_df = (
        pl.scan_csv(
            "CMAPSSData/train_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            new_columns=col_names,
            include_file_paths="file_path",
        )
        .select(col_names + ["file_path"])
        .with_columns(_dataset_id_expr())
        .with_columns(
            (pl.col("cycle").max().over(["dataset_id", "unit"]) - pl.col("cycle"))
            .clip(upper_bound=MAX_RUL)
            .alias("RUL")
        )
        .collect()
    )

    test_labels_df = (
        pl.scan_csv(
            "CMAPSSData/RUL_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            include_file_paths="file_path",
        )
        .with_columns(_dataset_id_expr())
        .sort(["dataset_id"], maintain_order=True)
        .with_columns(pl.int_range(1, pl.len() + 1).over("dataset_id").alias("unit"))
        .select("dataset_id", "unit", pl.col("column_1").alias("true_end_rul"))
        .collect()
    )

    test_df = (
        pl.scan_csv(
            "CMAPSSData/test_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            new_columns=col_names,
            include_file_paths="file_path",
        )
        .select(col_names + ["file_path"])
        .with_columns(_dataset_id_expr())
        .collect()
    )

    return train_df, test_df, test_labels_df

In [4]:
# ==========================================
# 2 Creat windows
# ==========================================

def attach_test_labels(test_df: pl.DataFrame, test_labels_df: pl.DataFrame) -> pl.DataFrame:
    """Return one row per test unit: its last recorded cycle + true_end_rul,
    joined by (dataset_id, unit) — never by position or raw file_path."""

    last_rows = (
        test_df.sort("cycle")
        .group_by(["dataset_id", "unit"], maintain_order=True)
        .last()  # last cycle per unit; keeps all original columns for that row
    )

    last_rows = last_rows.join(
        test_labels_df,
        on=["dataset_id", "unit"],
        how="left",
    )

    _validate_label_join(last_rows)
    return last_rows


def _validate_label_join(last_rows: pl.DataFrame):
    n_null = last_rows["true_end_rul"].null_count()
    if n_null > 0:
        missing = last_rows.filter(pl.col("true_end_rul").is_null()).select(
            "dataset_id", "unit"
        )
        raise ValueError(f"{n_null} test units have no matching RUL label:\n{missing}")

def train_windows(df, feature_cols, window=WINDOW_SEQ):

    X, y = [], []

    for _, group_df in df.group_by(["file_path", "unit"]):
        group_df = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        labels = group_df["RUL"].to_numpy()
        # print(CMAPSSData.shape, labels.shape)
        for i in range(len(group_df)-window+1):
            X.append(data[i:i+window])
            y.append(labels[i+window-1])
        # print(X.shape, y.shape)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def test_windows(test_df: pl.DataFrame, test_labels_df: pl.DataFrame, feature_cols, window=WINDOW_SEQ):
    labels_by_unit = attach_test_labels(test_df, test_labels_df)
    label_lookup = dict(
        zip(
            zip(labels_by_unit["dataset_id"], labels_by_unit["unit"]),
            labels_by_unit["true_end_rul"],
        )
    )

    X, y = [], []
    for (dataset_id, unit), group_df in test_df.group_by(["dataset_id", "unit"], maintain_order=True):
        group_df = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        if len(data) >= window:
            X.append(data[-window:])
        else:
            pad = np.zeros((window - len(data), len(feature_cols)))
            X.append(np.vstack([pad, data]))
        y.append(label_lookup[(dataset_id, unit)])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [5]:
# ==========================================
# 3 Create Datasets
# ==========================================

from torch.utils.data import Dataset, DataLoader

class Train_Dataset(Dataset):
    def __init__(self, X, y, augmentation=True):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
        self.augmentation = augmentation

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        x = self.X[index]
        if self.augmentation:
            x = x + torch.randn_like(x) * NOISE_STD
        return x, self.y[index]

class Test_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


In [6]:
# ==========================================
# 4 Model
# ==========================================
class Inception(nn.Module):
    def __init__(self, feature_input, feature_output, c1, c2, c3, use_residual):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Conv1d(feature_input, c1, kernel_size = 3, padding = 1),
            nn.BatchNorm1d(c1),
            nn.GELU()
        )
        self.b2 = nn.Sequential(
            nn.Conv1d(feature_input, c2, kernel_size = 7, padding = 3),
            nn.BatchNorm1d(c2),
            nn.GELU()
        )
        self.b3 = nn.Sequential(
            nn.Conv1d(feature_input, c3, kernel_size = 11, padding = 5),
            nn.BatchNorm1d(c3),
            nn.GELU()
        )
        self.sum_branches = nn.Sequential(
            nn.Conv1d(c1+c2+c3, feature_output, kernel_size = 1),
            nn.BatchNorm1d(feature_output),
        )
        # self.b4_0 = nn.MaxPool1d(kernel_size = 3, stride = 1, padding = 1)
        # self.b4_1 = nn.Conv1d(feature_input, c4, kernel_size = 11, padding = 5)
        if use_residual:
            self.residual = nn.Conv1d(feature_input, feature_output, kernel_size = 1)
            self.residual_bn = nn.BatchNorm1d(feature_output)

    def forward(self, x):
        b1 = self.b1(x)
        b2 = self.b2(x)
        b3 = self.b3(x)
        Y = torch.cat([b1, b2, b3], dim=1)
        Y = self.sum_branches(Y)
        X = 0
        if self.residual:
            X = self.residual_bn(self.residual(x))
        # b4 = self.b4_1(self.b4_0(x))
        return F.gelu(Y+X)


class LSTM(nn.Module):
    def __init__(self,
                 n_features,
                 hidden_size = HIDDEN_SIZE,
                 cnn_filters = CNN_FILTERS,
                 cnn_kernel = CNN_KERNEL,
                 c1 = C1,
                 c2 = C2,
                 c3 = C3,
                 use_residual = USE_RESIDUAL,
                 inception_out = INCEPTION_OUT,
                 mlp_dropout = MLP_DROPOUT,
                 lstm_dropout = LSTM_DROPOUT,
                 num_layers = NUM_LAYERS,
                 attention_head = ATTENTION_HEAD,
                 attention_dropout = ATTENTION_DROPOUT,
                 d_ff = D_FF,
                 ff_dropout = FF_DROPOUT,
                 ):
        super().__init__()

        # ======== 1D - CONV ==================
        # self.cnn = nn.Sequential(
        #     nn.Conv1d(in_channels=n_features, out_channels=cnn_filters, kernel_size=cnn_kernel, padding=cnn_kernel//2),
        #     nn.BatchNorm1d(cnn_filters),
        #     nn.GELU(),
        #     nn.Conv1d(in_channels=cnn_filters, out_channels=cnn_filters, kernel_size=cnn_kernel, padding=cnn_kernel//2),
        #     nn.BatchNorm1d(cnn_filters),
        #     nn.GELU(),
        #     nn.Conv1d(in_channels=cnn_filters, out_channels=cnn_filters*2, kernel_size=cnn_kernel, padding=cnn_kernel//2),
        #     nn.BatchNorm1d(cnn_filters*2),
        #     nn.GELU(),
        # )
        self.cnn_0 = nn.Sequential(
            Inception(n_features, inception_out, c1, c2, c3, use_residual),
            nn.Conv1d(in_channels=inception_out, out_channels=cnn_filters, kernel_size=cnn_kernel, padding=cnn_kernel // 2),
            nn.BatchNorm1d(cnn_filters),
            nn.GELU(),
        )

        # ======== LSTM ==================
        self.lstm = nn.LSTM(
            # input_size=cnn_filters*2,
            # input_size=Inception_CNN_IN,
            input_size = cnn_filters,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout = lstm_dropout if num_layers > 1 else 0.0
        )

        lstm_out = hidden_size * 2

        # ========= Attentnion  =====
        self.attn_norm = nn.LayerNorm(lstm_out)
        self.mha       = nn.MultiheadAttention(
            embed_dim  = lstm_out,
            num_heads  = attention_head,
            dropout    = attention_dropout,
            batch_first= True,
        )
        # Position-wise feed-forward (Pre-LN Transformer style)
        # self.ff_norm = nn.LayerNorm(lstm_out)
        # self.ff      = nn.Sequential(
        #     nn.Linear(lstm_out, d_ff),
        #     nn.GELU(),
        #     nn.Dropout(ff_dropout),
        #     nn.Linear(d_ff, lstm_out),
        # )
        # ======== regression - MLP ==================
        lstm_out = lstm_out *2
        self.regressor = nn.Sequential(
            nn.Linear(lstm_out, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(mlp_dropout),
            # nn.Linear(128, 64),
            # nn.LayerNorm(64),
            # nn.GELU(),
            # nn.Dropout(mlp_dropout * 0.5),
            nn.Linear(128, 1)

        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)

    def forward(self, x):
        #1D-CNN

        h = self.cnn_0(x.transpose(1, 2))   # (B, C, T)
        h = h.transpose(1, 2)             # (B, T, C)

        # BiLSTM
        lstm_out, (hn, _) = self.lstm(h)  # (B, T, 2H), hn: (2*L, B, H)
        # # Concatenate final forward + backward hidden states
        last_h = torch.cat([hn[-2], hn[-1]], dim=-1)  # (B, 2H)

        # Multi-head self-attention (Pre-LN)
        residual  = lstm_out
        normed    = self.attn_norm(lstm_out)
        attn_out, _ = self.mha(normed, normed, normed)
        lstm_out  = residual + attn_out

        # Position-wise feed-forward
        # residual  = lstm_out
        # normed    = self.ff_norm(lstm_out)
        # lstm_out  = residual + self.ff(normed)

        # Global average pool over time (attention-refined)
        mean_pool = lstm_out.mean(dim=1)   # (B, 2H)

        # Concatenate and regress
        out = torch.cat([last_h, mean_pool], dim=-1)  # (B, 4H)
        return self.regressor(out).squeeze(-1)         # (B,)

In [7]:
class GradientMonitor:
    """Monitors and records gradient norms across training iterations to detect

    exploding/vanishing gradients.
    """

    def __init__(self, model: nn.Module):
        self.model = model
        self.history: list[float] = []

    def compute_norm(self) -> float:
        """Computes and records the total L2 norm of the model's gradients."""
        total_sq_norm = 0.0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.detach().norm(2)
                total_sq_norm += param_norm.item() ** 2

        total_norm = total_sq_norm**0.5
        self.history.append(total_norm)
        return total_norm

    def print_summary(self) -> None:
        """Outputs statistical metrics for the captured gradient history."""
        if not self.history:
            print("[GradientMonitor] No gradients recorded yet.")
            return

        print(
            f"[Gradient Monitor] Mean: {np.mean(self.history):.6f} | "
            f"Max: {np.max(self.history):.6f} | Min: {np.min(self.history):.6f}"
        )

    def reset(self) -> None:
        """Clears stored history for a new epoch or evaluation."""
        self.history.clear()

In [8]:
# ==========================================
# 5 Trainer function
# ==========================================
def MSE(y_hat, y):
    return torch.mean((y_hat - y) ** 2)


class Trainer:
    def __init__(
        self,
        model: nn.Module,
        train_dataloader: DataLoader,
        test_dataloader: DataLoader,
        optimizer: torch.optim.Optimizer,
        scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None,
        device: torch.device | str = "cpu",
        loss_fn: Optional[Callable[[torch.Tensor, torch.Tensor], torch.Tensor]] = None,
        epochs: int = 10,
        grad_clip: Optional[float] = None,
        check_gradient: bool = False
     ):

        self.device = torch.device(device)
        self.model = model.to(self.device)
        self.train_dataloader = train_dataloader
        self.test_dataloader = test_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.loss_fn = loss_fn if loss_fn is not None else nn.MSELoss()
        self.epochs = epochs
        self.grad_clip = grad_clip
        self.grad_monitor = GradientMonitor(self.model) if check_gradient else None

    def fit_epoch(self) -> Tuple[float, float]:
        self.model.train()
        total_loss = 0.0
        total_samples = 0

        if self.grad_monitor:
            self.grad_monitor.reset()

        for X, y in self.train_dataloader:
            X, y = X.to(self.device), y.to(self.device)

            self.optimizer.zero_grad(set_to_none=True)

            pred = self.model(X).squeeze(-1)   # ensure shape (batch,)
            loss = self.loss_fn(pred, y)     # MSE
            loss.backward()

            if self.grad_monitor:
                self.grad_monitor.compute_norm()

            if self.grad_clip:
                nn.utils.clip_grad_norm_(model.parameters(),max_norm =  GRAD_CLIP)

            self.optimizer.step()

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size          # sum of MSE losses (weighted by batch size)

            total_samples += batch_size

        # Return average MSE and RMSE for the epoch (optional)
        avg_mse = total_loss / total_samples
        rmse = np.sqrt(avg_mse)

        if self.grad_monitor:
                self.grad_monitor.print_summary()

        return avg_mse, rmse

    def validate_epoch(self) -> float:
        self.model.eval()
        total_loss = 0.0
        total_samples = 0

        with torch.no_grad():
            for X, y in self.test_dataloader:
                X, y = X.to(self.device), y.to(self.device)
                pred = self.model(X).squeeze(-1)
                loss = self.loss_fn(pred, y)

                batch_size = y.size(0)
                total_loss += loss.item() * batch_size
                total_samples += batch_size

        rmse = np.sqrt(total_loss / total_samples)
        return rmse

    def fit(self):
        for epoch in range(self.epochs):
            train_mse, train_rmse = self.fit_epoch()
            val_rmse = self.validate_epoch()

            if self.scheduler:
                self.scheduler.step()

            print(f"Epoch {epoch+1}/{self.epochs} | Train MSE: {train_mse:.4f} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

In [9]:
# Load
if __name__ == "__main__":

    train_df, test_df, RUL_test = load_all_data()
    train_X, train_y = train_windows(train_df, feature_cols)
    test_X, test_y = test_windows(test_df, RUL_test, feature_cols)

    scaler = StandardScaler().fit(train_X.reshape(-1, len(feature_cols)))
    train_X = scaler.transform(train_X.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))
    test_X  = scaler.transform(test_X.reshape(-1, len(feature_cols))).reshape(-1, WINDOW_SEQ, len(feature_cols))

    train_dataloader = DataLoader(Train_Dataset(train_X, train_y), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_OF_WORKERS, pin_memory=True, drop_last=True)
    test_dataloader = DataLoader(Test_Dataset(test_X, test_y), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_OF_WORKERS, pin_memory=True)

    loss_fn = MSE
    # model = LSTM(n_features=len(feature_cols), hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, cnn_filters=CNN_FILTERS, cnn_kernel=CNN_KERNEL, lstm_bidirectional=True, lstm_dropout = 0.2, mlp_dropout=0.4)
    model = LSTM(len(feature_cols))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(trainable_params, total_params  )
    # model.to(device)

    optimiser = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    # Cosine Annealing with Warm Restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimiser, T_0=T0, T_mult=T_MULT, eta_min=LR * 1e-6
    )

    trainer = Trainer(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        optimizer=optimiser,
        scheduler=scheduler,
        device=device,
        loss_fn=loss_fn,
        epochs=EPOCHS,
        check_gradient=False,
        grad_clip=1.0
    )

    trainer.fit()

435321 435321
Epoch 1/60 | Train MSE: 2708.2537 | Train RMSE: 52.0409 | Val RMSE: 29.8714
Epoch 2/60 | Train MSE: 629.4094 | Train RMSE: 25.0880 | Val RMSE: 26.8792
Epoch 3/60 | Train MSE: 539.5673 | Train RMSE: 23.2286 | Val RMSE: 26.7367
Epoch 4/60 | Train MSE: 486.6705 | Train RMSE: 22.0606 | Val RMSE: 26.3265
Epoch 5/60 | Train MSE: 427.3277 | Train RMSE: 20.6719 | Val RMSE: 26.9148
Epoch 6/60 | Train MSE: 365.1528 | Train RMSE: 19.1090 | Val RMSE: 25.9403
Epoch 7/60 | Train MSE: 312.4161 | Train RMSE: 17.6753 | Val RMSE: 25.6418
Epoch 8/60 | Train MSE: 271.1662 | Train RMSE: 16.4671 | Val RMSE: 25.1114
Epoch 9/60 | Train MSE: 238.9372 | Train RMSE: 15.4576 | Val RMSE: 25.5020
Epoch 10/60 | Train MSE: 215.2864 | Train RMSE: 14.6726 | Val RMSE: 25.1940
Epoch 11/60 | Train MSE: 196.4676 | Train RMSE: 14.0167 | Val RMSE: 26.5090
Epoch 12/60 | Train MSE: 184.3892 | Train RMSE: 13.5790 | Val RMSE: 25.6229
Epoch 13/60 | Train MSE: 174.8337 | Train RMSE: 13.2225 | Val RMSE: 25.9424
Epoch 

In [10]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(trainable_params, total_params, model.parameters())

435321 435321 <generator object Module.parameters at 0x0000019F8AD26C00>


In [11]:
for param in model.parameters():
    print(param.size())

torch.Size([34, 14, 3])
torch.Size([34])
torch.Size([34])
torch.Size([34])
torch.Size([20, 14, 7])
torch.Size([20])
torch.Size([20])
torch.Size([20])
torch.Size([10, 14, 11])
torch.Size([10])
torch.Size([10])
torch.Size([10])
torch.Size([82, 64, 1])
torch.Size([82])
torch.Size([82])
torch.Size([82])
torch.Size([82, 14, 1])
torch.Size([82])
torch.Size([82])
torch.Size([82])
torch.Size([80, 82, 3])
torch.Size([80])
torch.Size([80])
torch.Size([80])
torch.Size([320, 80])
torch.Size([320, 80])
torch.Size([320])
torch.Size([320])
torch.Size([320, 80])
torch.Size([320, 80])
torch.Size([320])
torch.Size([320])
torch.Size([320, 160])
torch.Size([320, 80])
torch.Size([320])
torch.Size([320])
torch.Size([320, 160])
torch.Size([320, 80])
torch.Size([320])
torch.Size([320])
torch.Size([160])
torch.Size([160])
torch.Size([480, 160])
torch.Size([480])
torch.Size([160, 160])
torch.Size([160])
torch.Size([128, 320])
torch.Size([128])
torch.Size([128])
torch.Size([128])
torch.Size([1, 128])
torch.Size(

In [12]:
x = torch.randn(size=(64, 32, 19), requires_grad=False)


In [13]:
x.shape

torch.Size([64, 32, 19])

In [14]:
model = Inception(19, 12, 8, 3, 5)

TypeError: Inception.__init__() missing 1 required positional argument: 'use_residual'

In [ ]:
model

In [ ]:
x.transpose(1,2).shape

In [ ]:
model(x.transpose(1,2))

In [ ]:
 # train_X.shape, train_y.shape,    test_X.shape, test_y.shape